# Handover: train SmolVLA on a free Colab GPU

Trains the first real SmolVLA policy for the cup-on-tray task, then lets it play in the game's own simulation and renders a video.

1. **Runtime → Change runtime type → T4 GPU** (any GPU works).
2. **Runtime → Run all.** It takes a few hours on a free T4. Keep the tab open.
3. At the end, download `handover_results.zip`. It holds the trained model, the video of it playing, and the success rate.

Change the settings in the next cell if you like.

In [ ]:
BRANCH = "claude/project-thread-5ruxby"  # training pipeline branch (use "main" once merged)
DEMOS = 50          # scripted demos to train on (SmolVLA authors: ~50 is the minimum)
STEPS = 5000        # training steps; 20000 is the full recipe, ~4x longer
BATCH = 32          # fits a 16 GB T4; use 64 on an A100
ROLLOUTS = 10       # episodes the trained model plays for the success rate

In [ ]:
%%bash -s "$BRANCH"
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
rm -rf handover && git clone -q -b "$1" https://github.com/syldrm3141/handover.git
pip -q install "lerobot[smolvla,training]==0.6.1" "mujoco>=3.3" 2>&1 | tail -2
if ! node -e 'process.exit(+process.versions.node.split(".")[0] < 18)' 2>/dev/null; then
  curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1
fi
node --version

In [ ]:
import os
os.chdir("/content/handover/training")
os.environ["MUJOCO_GL"] = "egl"  # GPU rendering for the camera views
!python -m pytest -q tests

## 1. Demos and dataset

In [ ]:
!node scripted_demos.mjs --out data/raw --episodes {DEMOS} --seed 0 | tail -3
!python validate.py data/raw
!python -u convert.py data/raw --out data/lerobot/handover_kitchen --overwrite 2>&1 | grep --line-buffered "^\\[\\|wrote\\|skipped\\|note:"
!python validate.py --dataset data/lerobot/handover_kitchen

## 2. Fine-tune SmolVLA

In [ ]:
!DEVICE=cuda BATCH={BATCH} STEPS={STEPS} SAVE_FREQ={STEPS} WORKERS=2 ./train_smolvla.sh --log_freq=100 2>&1 | grep -v "it/s" | tail -40

## 3. Watch it play

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(
    ["python", "policy_server.py", "--device", "cuda",
     "--policy", "outputs/train/smolvla_handover/checkpoints/last/pretrained_model"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8765/info").read().decode()); break
    except Exception:
        time.sleep(5)
!node rollout.mjs --task cup_on_tray --episodes {ROLLOUTS} --out runs/smolvla.jsonl.gz | tee runs/success_rate.txt
server.terminate()

In [ ]:
!python preview.py runs/smolvla.jsonl.gz --max-episodes 4 --out runs/smolvla_playing.mp4
from IPython.display import Video
Video("runs/smolvla_playing.mp4", embed=True, width=800)

## 4. Download the results

In [ ]:
!cd outputs/train/smolvla_handover/checkpoints/last && zip -qr /content/handover_results.zip pretrained_model
!cd runs && zip -qr /content/handover_results.zip smolvla_playing.mp4 success_rate.txt smolvla.jsonl.gz
from google.colab import files
files.download("/content/handover_results.zip")